# 5G NR Multicell Twin — Sionna RT on open San Francisco data (Colab T4)

Reproduces every ray trace in the paper *Open Reproducible Ray-Tracing Pipeline for 5G NR Multicell Digital Twins*: three sites × three sectors at 3.6 GHz over a 1 km × 1 km tile at Telegraph Hill / North Beach (centre 37.801105, −122.404175), traced with Sionna RT 2.2.0 on an open city scene built from City of San Francisco building footprints (DataSF, CC0).

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**Run it on your own storage** — nothing is read from the authors' accounts:
1. Run all. Step 3 uses `city_open.ply` and `SF_sites.csv` if they are next to the notebook (in Colab: the Files pane), and otherwise downloads them once from the repository's `data/` folder. To work fully offline, download them first, or rebuild the scene with `scripts/build_scene.py`.
2. Results are written to `./out`. On a T4 the whole notebook takes a few minutes.

Steps 6–11 produce the paper's results: 6 = full tile (Table II), 7 = tower-B window (Fig. 3), 8 = sensitivity runs (Table IV), 9 = tower-shift sweep (Table III), 10 = robustness checks (ray-sampling seeds, terrain face orientation), 11 = summary.

In [ ]:
# 1. Check the T4
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import subprocess; assert 'T4' in subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or print('WARNING: no T4 - switch the runtime type to T4 GPU')

In [ ]:
# 2. Install Sionna RT (pinned: ray-tracing results are version-sensitive) + mesh tools
!pip -q install sionna-rt==2.2.0 trimesh scipy
import mitsuba as mi, sionna.rt as rt
print('Sionna RT', rt.__version__, '| variant', mi.variant())  # expect cuda_ad_mono_polarized

In [ ]:
# 3. Data source
#  'local'  = reader mode: use ./city_open.ply and ./SF_sites.csv next to this notebook;
#             any file not found is downloaded once from the public repository's data/ folder
#  'upload' = upload both files into this Colab session
DATA_SOURCE = 'local'
import os, urllib.request
REPO_DATA = 'https://raw.githubusercontent.com/ysrao/twin5GNR-multicell-SF/main/data/'
if DATA_SOURCE == 'upload':
    from google.colab import files
    files.upload()   # choose city_open.ply and SF_sites.csv
SCENE_FILE, CSV_PATH, OUT_DIR = './city_open.ply', './SF_sites.csv', './out'
for p in (SCENE_FILE, CSV_PATH):
    if not os.path.exists(p):
        print('downloading', os.path.basename(p), 'from the repository')
        urllib.request.urlretrieve(REPO_DATA + os.path.basename(p), p)
os.makedirs(OUT_DIR, exist_ok=True)
print('DATA_SOURCE =', DATA_SOURCE, '| scene', SCENE_FILE, '| sites', CSV_PATH, '| out', OUT_DIR)

In [ ]:
%%writefile twin_rt.py
"""5G NR Multicell Twin - Sionna RT run on the open San Francisco scene.

Produces per-cell, per-sector RSRP on the street grid
(25 x 25 at 40 m, ground + 1.5 m, 3 towers x 3 sectors, 3.6 GHz).
Designed for a Colab T4 (cuda_ad_mono_polarized); also runs on CPU (llvm).
"""
import os, json, glob, time, math
import numpy as np

# ---------------------------------------------------------------- config
CFG = dict(
    MESH_PATH="./scene",     # folder or file: .xml, .obj, .ply, .glb, .gltf
    OUT_DIR="./out",
    UP_AXIS="auto",          # "z", "y" or "auto"
    ORIGIN_XY=None,          # mesh coords of the tile centre; None = centre of mesh bbox
    CROP_HALF_M=700.0,       # keep geometry within +/- this of the centre (tile 500 + margin)
    FREQ_HZ=3.6e9,
    ITU_MATERIAL="concrete", THICKNESS_M=0.2, SCATTERING=0.3,
    TILE_HALF_M=500.0,
    GRID_N=25, GRID_STEP_M=40.0, RX_H_M=1.5,
    # tower positions in local metres (x east, y north, tile centre = 0,0)
    TOWERS={"A": (-250.0, 250.0), "B": (250.0, 250.0), "C": (0.0, -250.0)},
    TOWER_H_M=30.0,                      # above local ground
    SECTOR_AZ_DEG=(90.0, 210.0, 330.0),  # compass bearing, clockwise from north
    TILT_DEG=None,                       # None = auto 0-10 deg per sector, or a number
    EIRP_DBM=47.0, ELEMENT_GAIN_DBI=8.0, # tr38901 element peak gain inside the solver
    BW_HZ=100e6, SCS_HZ=30e3,            # for RSRP per RE
    MAX_DEPTH=3, SAMPLES_PER_SRC=int(1e6), SEED=42,  # PathSolver seed (Sionna default 42)
    LOS=True, SPECULAR=True, DIFFUSE=True, REFRACTION=False, DIFFRACTION=True,
    DEMO_SCENE=None,                     # e.g. "simple_street_canyon" to smoke-test without your mesh
)

# ---------------------------------------------------------------- mesh -> Mitsuba scene
XML_TMPL = """<scene version="2.1.0">
  <bsdf type="itu-radio-material" id="mat-itu_{mat}">
    <string name="type" value="{mat}"/>
    <float name="thickness" value="{th}"/>
    <float name="scattering_coefficient" value="{sc}"/>
  </bsdf>
  <shape type="ply" id="city">
    <string name="filename" value="meshes/city.ply"/>
    <ref id="mat-itu_{mat}" name="bsdf"/>
  </shape>
</scene>
"""

def prepare_scene(cfg):
    """Return path of a Mitsuba XML scene in local metric coords centred on the tile."""
    p = cfg["MESH_PATH"]
    if os.path.isdir(p):
        xml = sorted(glob.glob(os.path.join(p, "*.xml")))
        if xml:
            print("Using existing Mitsuba scene:", xml[0]); return xml[0], None
        files = [f for e in ("obj", "ply", "glb", "gltf", "stl", "OBJ", "PLY", "GLB")
                 for f in glob.glob(os.path.join(p, "**", "*." + e), recursive=True)]
        if not files: raise FileNotFoundError(f"No mesh files under {p}")
    elif p.lower().endswith(".xml"):
        return p, None
    else:
        files = [p]
    import trimesh
    parts = []
    for f in files:
        m = trimesh.load(f, force="mesh", process=False)
        if len(m.faces): parts.append(m)
    mesh = trimesh.util.concatenate(parts)
    v = np.asarray(mesh.vertices, dtype=np.float64)
    ext = v.max(0) - v.min(0)
    print(f"Loaded {len(files)} file(s): {len(mesh.faces):,} triangles, extents {np.round(ext,1)}")
    if ext.max() < 1.0:
        raise ValueError("Mesh extents < 1 unit: looks like degrees (lat/lon). Reproject to UTM metres first.")
    up = cfg["UP_AXIS"]
    if up == "auto":  # vertical axis = smallest extent for a city tile
        up = "y" if np.argmin(ext) == 1 else "z"
    if up == "y":     # Y-up -> Z-up (x east, y north = -z_gltf, z up)
        v = np.stack([v[:, 0], -v[:, 2], v[:, 1]], 1)
    ox, oy = cfg["ORIGIN_XY"] if cfg["ORIGIN_XY"] else ((v[:, 0].min() + v[:, 0].max()) / 2,
                                                          (v[:, 1].min() + v[:, 1].max()) / 2)
    v[:, 0] -= ox; v[:, 1] -= oy
    f = np.asarray(mesh.faces)
    c = v[f].mean(1); h = cfg["CROP_HALF_M"]
    keep = (np.abs(c[:, 0]) <= h) & (np.abs(c[:, 1]) <= h)
    f = f[keep]
    print(f"Up axis {up}; origin ({ox:.1f},{oy:.1f}); kept {keep.sum():,} triangles within +/-{h:.0f} m; "
          f"z range {v[:,2].min():.1f}..{v[:,2].max():.1f} m")
    out = cfg["OUT_DIR"]; os.makedirs(os.path.join(out, "scene", "meshes"), exist_ok=True)
    trimesh.Trimesh(v.astype(np.float32), f, process=True).export(os.path.join(out, "scene", "meshes", "city.ply"))
    xml = os.path.join(out, "scene", "city.xml")
    open(xml, "w").write(XML_TMPL.format(mat=cfg["ITU_MATERIAL"], th=cfg["THICKNESS_M"], sc=cfg["SCATTERING"]))
    return xml, (ox, oy)

# ---------------------------------------------------------------- geometry helpers
def surface_z(mi_scene, xy, z0=2000.0):
    """Height of the first surface hit straight down at each (x,y); nan if none."""
    import mitsuba as mi, drjit as dr
    xy = np.asarray(xy, np.float32)
    o = mi.Point3f(xy[:, 0], xy[:, 1], np.full(len(xy), z0, np.float32))
    d = mi.Vector3f(0, 0, -1)
    si = mi_scene.ray_intersect(mi.Ray3f(o, d))
    t = np.array(si.t); ok = np.array(si.is_valid())
    return np.where(ok, z0 - t, np.nan)

def street_grid(mi_scene, cfg):
    """Receiver grid: open cells at ground+RX_H, in-building points nudged to nearest open spot."""
    H = cfg["CROP_HALF_M"]; s = 4.0
    xs = np.arange(-H + s / 2, H, s)
    X, Y = np.meshgrid(xs, xs)
    Z = surface_z(mi_scene, np.c_[X.ravel(), Y.ravel()]).reshape(X.shape)
    from scipy.ndimage import percentile_filter
    Zf = np.where(np.isnan(Z), np.nanmin(Z), Z)
    ground = percentile_filter(Zf, 5, size=int(60 / s))      # local ground: 5th pct in 60 m window
    openm = (Zf - ground) < 2.5
    n, st = cfg["GRID_N"], cfg["GRID_STEP_M"]
    g = (np.arange(n) - (n - 1) / 2) * st
    cells, nudged = [], 0
    oy, ox = np.nonzero(openm)
    for r in range(n):                     # rows north -> south, cols west -> east
        y = g[::-1][r]
        for cidx in range(n):
            x = g[cidx]
            j = int(np.argmin(np.abs(xs - x))); i = int(np.argmin(np.abs(xs - y)))
            if not openm[i, j]:
                k = np.argmin((ox - j) ** 2 + (oy - i) ** 2); i, j = oy[k], ox[k]; nudged += 1
            cells.append((float(xs[j]), float(xs[i]), float(Zf[i, j] + cfg["RX_H_M"])))
    print(f"Grid {n}x{n}: {nudged} receivers nudged out of buildings")
    return np.array(cells), (xs, ground, Zf)

def tower_sites(cfg, terrain):
    xs, ground, Zf = terrain
    sites = {}
    for name, (x, y) in cfg["TOWERS"].items():
        j = int(np.argmin(np.abs(xs - x))); i = int(np.argmin(np.abs(xs - y)))
        g = float(ground[i, j]); w = int(30 / (xs[1] - xs[0]))
        roof = float(np.nanmax(Zf[max(0, i - w):i + w + 1, max(0, j - w):j + w + 1]))
        z = g + cfg["TOWER_H_M"]
        sites[name] = dict(x=x, y=y, ground=g, z=z, clear_local_roof_m=round(z - roof, 1))
    return sites

def auto_tilt(site, cells, az_deg):
    """Tilt aiming at the median distance of cells inside this sector's 120 deg wedge, clamped 0-10 deg."""
    dx, dy = cells[:, 0] - site["x"], cells[:, 1] - site["y"]
    bear = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
    inw = np.abs(((bear - az_deg + 180) % 360) - 180) <= 60
    d = np.median(np.hypot(dx, dy)[inw]) if inw.any() else 300.0
    dz = site["z"] - np.median(cells[inw, 2]) if inw.any() else 28.5
    return float(np.clip(np.degrees(np.arctan2(dz, d)), 0, 10))

# ---------------------------------------------------------------- main run
def run(cfg=CFG):
    import mitsuba as mi, drjit as dr
    import sionna.rt as rt
    from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver
    print("Sionna RT", rt.__version__, "| Mitsuba variant", mi.variant())
    t0 = time.time()
    if cfg["DEMO_SCENE"]:
        scene = load_scene(getattr(rt.scene, cfg["DEMO_SCENE"])); origin = None
    else:
        xml, origin = prepare_scene(cfg); scene = load_scene(xml)
    scene.frequency = cfg["FREQ_HZ"]
    scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="tr38901", polarization="V")
    scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

    cells, terrain = street_grid(scene.mi_scene, cfg)
    sites = tower_sites(cfg, terrain)
    sectors = []
    for tn, s in sites.items():
        for az in cfg["SECTOR_AZ_DEG"]:
            tilt = cfg["TILT_DEG"] if cfg["TILT_DEG"] is not None else auto_tilt(s, cells, az)
            sectors.append(dict(name=f"{tn}{int(az)}", tower=tn, az=az, tilt=round(tilt, 2)))
    for k, (x, y, z) in enumerate(cells):
        scene.add(Receiver(f"rx{k}", position=mi.Point3f(float(x), float(y), float(z))))

    solver = PathSolver()
    G = np.full((len(cells), len(sectors)), np.nan)   # path gain dB (includes element pattern)
    for si, sec in enumerate(sectors):                 # one sector at a time keeps T4 memory low
        s = sites[sec["tower"]]
        yaw = math.radians(90.0 - sec["az"]); pitch = math.radians(sec["tilt"])
        tx = Transmitter("tx", position=mi.Point3f(s["x"], s["y"], s["z"]),
                         orientation=mi.Point3f(yaw, pitch, 0.0))
        scene.add(tx)
        t1 = time.time()
        paths = solver(scene, max_depth=cfg["MAX_DEPTH"], samples_per_src=cfg["SAMPLES_PER_SRC"],
                       los=cfg["LOS"], specular_reflection=cfg["SPECULAR"],
                       diffuse_reflection=cfg["DIFFUSE"], refraction=cfg["REFRACTION"],
                       diffraction=cfg["DIFFRACTION"], synthetic_array=True,
                       seed=cfg.get("SEED", 42))
        a_re, a_im = paths.a
        p = (np.array(a_re) ** 2 + np.array(a_im) ** 2)  # [rx, rx_ant, tx, tx_ant, paths]
        g = p.reshape(p.shape[0], -1).sum(1)
        G[:, si] = np.where(g > 0, 10 * np.log10(np.maximum(g, 1e-30)), np.nan)
        scene.remove("tx")
        print(f"  sector {sec['name']:>5} tilt {sec['tilt']:4.1f} deg: {np.isfinite(G[:,si]).mean()*100:5.1f}% "
              f"cells with a path, {time.time()-t1:5.1f} s")

    p_cond = cfg["EIRP_DBM"] - cfg["ELEMENT_GAIN_DBI"]
    prx = p_cond + G                                   # total received power, dBm
    n_re = 12 * int(cfg["BW_HZ"] / (12 * cfg["SCS_HZ"]) * 0.98)  # approx. usable RE
    rsrp = prx - 10 * np.log10(n_re)
    best = np.nanmax(np.where(np.isfinite(rsrp), rsrp, -np.inf), 1)
    best = np.where(np.isfinite(best), best, np.nan)

    def r(v): return None if not np.isfinite(v) else round(float(v), 2)
    out = dict(
        version="twin5GNR-multicell open scene / Sionna RT (Colab)",
        sionna_rt=rt.__version__, variant=mi.variant(), freq_hz=cfg["FREQ_HZ"],
        solver=dict(max_depth=cfg["MAX_DEPTH"], samples_per_src=cfg["SAMPLES_PER_SRC"], los=cfg["LOS"],
                    specular=cfg["SPECULAR"], diffuse=cfg["DIFFUSE"], diffraction=cfg["DIFFRACTION"],
                    material=f"itu_{cfg['ITU_MATERIAL']}", scattering=cfg["SCATTERING"]),
        mesh_origin_xy=origin, grid=dict(n=cfg["GRID_N"], step_m=cfg["GRID_STEP_M"],
                                         order="row-major, rows north->south, cols west->east"),
        towers=sites, sectors=sectors, n_re=n_re,
        cells=[dict(x=round(c[0], 1), y=round(c[1], 1), z=round(c[2], 1)) for c in cells],
        rsrp=[[r(v) for v in row] for row in rsrp],      # [cell][sector], dBm per RE
        rsrp_dbm=[r(v) for v in best],                   # best-server RSRP per cell
        best_server=[None if not np.isfinite(b) else sectors[int(np.nanargmax(row))]["name"]
                     for b, row in zip(best, np.where(np.isfinite(rsrp), rsrp, -np.inf))],
    )
    os.makedirs(cfg["OUT_DIR"], exist_ok=True)
    fn = os.path.join(cfg["OUT_DIR"], f"sionna_rsrp_{cfg['FREQ_HZ']/1e9:.1f}GHz.json")
    json.dump(out, open(fn, "w"))
    fin = best[np.isfinite(best)]
    print(f"\nDone in {time.time()-t0:.0f} s -> {fn}")
    print(f"cells with a path: {len(fin)}/{len(best)}; median best RSRP {np.median(fin):.1f} dBm; "
          f"coverage (>= -100 dBm) {(fin >= -100).sum()/len(best)*100:.1f}%")
    for n, s in sites.items():
        print(f"tower {n}: ground {s['ground']:.1f} m, antenna {s['z']:.1f} m, clears local roof by {s['clear_local_roof_m']} m")
    return out, fn

if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "demo":
        CFG.update(DEMO_SCENE="simple_street_canyon", OUT_DIR="./demo_out", CROP_HALF_M=120.0,
                   TOWERS={"A": (-40.0, 20.0)}, GRID_N=9, GRID_STEP_M=20.0, SAMPLES_PER_SRC=int(2e5))
    run(CFG)


## Changing the tilt

The tilt is set in the next cell, in the `T.CFG.update(...)` line: **`TILT_DEG=None`** means auto tilt (each sector aims at the median distance of its receivers, clamped to 0-10 deg — the paper's baseline). To run a **fixed tilt**, replace `None` with a number of degrees, the same for all nine sectors — e.g. `TILT_DEG=8.0`.

Then re-run the next cell and the full-tile cell below it (about 15 s on a T4), and run the **map cell at the end of this notebook** to see the coverage map your run produced. Cell 9 already runs the paper's fixed-4 deg sensitivity set if you want the published comparison instead of your own.

In [ ]:
# 5. Load the open scene and set the paper's configuration (Table I)
import importlib, os, shutil, copy, json, time
import numpy as np, trimesh
import twin_rt as T; importlib.reload(T)
SCENE_DIR = './scene_open'
os.makedirs(SCENE_DIR, exist_ok=True)
shutil.copyfile(SCENE_FILE, os.path.join(SCENE_DIR, 'city.ply'))
m = trimesh.load(os.path.join(SCENE_DIR, 'city.ply'), force='mesh', process=False)
print(f'triangles: {len(m.faces):,} (expected 110,396)')
assert len(m.faces) == 110396
AS_BUILT = {'A': (-250.0, 250.0), 'B': (250.0, 250.0), 'C': (0.0, -250.0)}
T.CFG.update(MESH_PATH=SCENE_DIR, OUT_DIR=OUT_DIR, FREQ_HZ=3.6e9, TOWERS=dict(AS_BUILT),
             TOWER_H_M=30.0, EIRP_DBM=47.0, TILT_DEG=None, SAMPLES_PER_SRC=int(1e6), MAX_DEPTH=3,
             DIFFUSE=True, DIFFRACTION=True, GRID_N=25, GRID_STEP_M=40.0, RX_H_M=1.5)
BASE = copy.deepcopy(T.CFG)

def save(out, name, **run):
    out['run'] = dict(label=name, scene='open', scene_file='city_open.ply', **run)
    fn = os.path.join(OUT_DIR, f'{name}.json')
    json.dump(out, open(fn, 'w'))
    b = np.array([np.nan if v is None else v for v in out['rsrp_dbm']], dtype=float)
    print(f"{name}: {np.isfinite(b).mean()*100:.1f}% with a path | median {np.nanmedian(b):.1f} dBm | "
          f"coverage >= -90 dBm {(b >= -90).mean()*100:.1f}% | saved {fn}")
    return out

def roof_plus(h):
    """tower_sites variant: antenna at local roof (max surface within +/-30 m) + h metres."""
    def sites(cfg, terrain):
        xs, ground, Zf = terrain
        out = {}
        for name, (x, y) in cfg['TOWERS'].items():
            j = int(np.argmin(np.abs(xs - x))); i = int(np.argmin(np.abs(xs - y)))
            g = float(ground[i, j]); w = int(30 / (xs[1] - xs[0]))
            roof = float(np.nanmax(Zf[max(0, i - w):i + w + 1, max(0, j - w):j + w + 1]))
            out[name] = dict(x=x, y=y, ground=g, z=roof + h, roof_m=round(roof, 1), clear_local_roof_m=h)
        return out
    return sites
print('configuration set:', {k: T.CFG[k] for k in ('FREQ_HZ', 'EIRP_DBM', 'TOWER_H_M', 'MAX_DEPTH', 'SAMPLES_PER_SRC', 'GRID_N', 'GRID_STEP_M')})

In [ ]:
# 6. Full tile, as-built (Table II baseline): auto tilt, 25 x 25 street grid at 40 m
t0 = time.time()
out_base, _ = T.run(copy.deepcopy(BASE))
out_base = save(out_base, 'open_full_tile', tilt_mode='auto_per_sector', wall_clock_s=round(time.time() - t0, 1))
for n, s in out_base['towers'].items():
    print(f"  tower {n}: ground {s['ground']:.1f} m, antenna {s['z']:.1f} m, clears local roof by {s['clear_local_roof_m']} m")

In [ ]:
# 7. Tower-B window (Fig. 3): 27 x 27 points at 10 m over x, y = 120..380 m, same tilts as step 6
from scipy.ndimage import percentile_filter
tilts = {s['name']: s['tilt'] for s in out_base['sectors']}
by_xy = {xy: n for n, xy in AS_BUILT.items()}
def tilt_from_base(site, cells, az_deg):
    return tilts[f"{by_xy[(site['x'], site['y'])]}{int(az_deg)}"]
def window_grid(mi_scene, cfg):
    H = cfg['CROP_HALF_M']; s = 4.0
    xs = np.arange(-H + s / 2, H, s)
    X, Y = np.meshgrid(xs, xs)
    Z = T.surface_z(mi_scene, np.c_[X.ravel(), Y.ravel()]).reshape(X.shape)
    Zf = np.where(np.isnan(Z), np.nanmin(Z), Z)
    ground = percentile_filter(Zf, 5, size=int(60 / s))
    openm = (Zf - ground) < 2.5
    oy, ox = np.nonzero(openm)
    cells, nudged = [], 0
    for y in np.arange(380, 120 - 1, -10, dtype=float):
        for x in np.arange(120, 380 + 1, 10, dtype=float):
            j = int(np.argmin(np.abs(xs - x))); i = int(np.argmin(np.abs(xs - y)))
            if not openm[i, j]:
                k = np.argmin((ox - j) ** 2 + (oy - i) ** 2); i, j = oy[k], ox[k]; nudged += 1
            cells.append((float(xs[j]), float(xs[i]), float(Zf[i, j] + cfg['RX_H_M'])))
    print(f'window 27x27: {nudged} receivers nudged out of buildings')
    return np.array(cells), (xs, ground, Zf)
orig_grid, orig_tilt = T.street_grid, T.auto_tilt
T.street_grid, T.auto_tilt = window_grid, tilt_from_base
try:
    t0 = time.time()
    cfg = copy.deepcopy(BASE); cfg.update(GRID_N=27, GRID_STEP_M=10.0)
    out_win, _ = T.run(cfg)
finally:
    T.street_grid, T.auto_tilt = orig_grid, orig_tilt
out_win['grid'] = dict(n_x=27, n_y=27, n=729, step_m=10.0, x_m=[120, 380], y_m=[120, 380],
                       order='row-major, rows north->south, cols west->east')
save(out_win, 'open_towerB_window_10m', tilt_mode='auto_per_sector_from_full_tile', wall_clock_s=round(time.time() - t0, 1))

In [ ]:
# 8. Sensitivity (Table IV): fixed 4 deg tilt; roof + 10 m antennas; roof + 10 m at solver depth 5
orig_sites = T.tower_sites
cfg = copy.deepcopy(BASE); cfg.update(TILT_DEG=4.0)
save(T.run(cfg)[0], 'open_tilt4', tilt_mode='fixed_4deg')
T.tower_sites = roof_plus(10.0)
try:
    cfg = copy.deepcopy(BASE); cfg.update(TILT_DEG=4.0)
    save(T.run(cfg)[0], 'open_tilt4_roofref', tilt_mode='fixed_4deg', antenna_height='local_roof_plus_10m')
    cfg = copy.deepcopy(BASE); cfg.update(TILT_DEG=4.0, MAX_DEPTH=5)
    save(T.run(cfg)[0], 'open_tilt4_roofref_depth5', tilt_mode='fixed_4deg', antenna_height='local_roof_plus_10m')
finally:
    T.tower_sites = orig_sites

In [ ]:
# 9. Tower-shift sweep (Table III): one tower moved 125 m E/W/N/S at its SF_sites.csv height,
# the other two stay as-built at 30 m. Change CASES to run a subset in class.
import csv, math
LAYOUT = 'macro-3.6GHz'
CASES = ['A-E', 'A-W', 'A-N', 'A-S', 'B-E', 'B-W', 'B-N', 'B-S', 'C-E', 'C-W', 'C-N', 'C-S']
LAT0, LON0 = 37.801105, -122.404175
def to_enu(lat, lon):
    return (round(math.radians(lon - LON0) * 6371000.0 * math.cos(math.radians(LAT0)), 1),
            round(math.radians(lat - LAT0) * 6371000.0, 1))
rows = {(r['layout'], r['site']): r for r in csv.DictReader(open(CSV_PATH))}
HEIGHTS = {}
def per_tower_sites(cfg, terrain):
    sites = orig_sites(cfg, terrain)
    for n, s in sites.items():
        roof = s['z'] - s['clear_local_roof_m']
        s['z'] = s['ground'] + HEIGHTS.get(n, cfg['TOWER_H_M'])
        s['clear_local_roof_m'] = round(s['z'] - roof, 1)
    return sites
T.tower_sites = per_tower_sites
sweep = []
try:
    for case in CASES:
        r = rows[(LAYOUT, case)]; tower = case.split('-')[0]
        HEIGHTS.clear(); HEIGHTS.update({t: (float(r['height_m_AGL']) if t == tower else 30.0) for t in AS_BUILT})
        cfg = copy.deepcopy(BASE); cfg['TOWERS'] = dict(AS_BUILT)
        cfg['TOWERS'][tower] = to_enu(float(r['latitude']), float(r['longitude']))
        out = save(T.run(cfg)[0], f'open_shift_{case}', heights_m_AGL=dict(HEIGHTS), tilt_mode='auto_per_sector')
        b = np.array([np.nan if v is None else v for v in out['rsrp_dbm']], dtype=float)
        sweep.append((case, HEIGHTS[tower], np.isfinite(b).mean() * 100, np.nanmedian(b), (b >= -90).mean() * 100))
finally:
    T.tower_sites = orig_sites
print(f"{'case':6}{'h (m)':>7}{'path %':>9}{'median dBm':>12}{'cov %':>8}")
for c, h, p, med, cov in sweep:
    print(f'{c:6}{h:7.0f}{p:9.1f}{med:12.1f}{cov:8.1f}')

In [ ]:
# 10. Robustness checks: (a) the baseline with three more PathSolver seeds, to bound
# Monte Carlo ray-sampling noise; (b) the baseline with every downward-facing triangle
# re-wound to face up (the terrain mesh is built with downward normals).
for seed in (1, 2, 3):
    cfg = copy.deepcopy(BASE); cfg['SEED'] = seed
    save(T.run(cfg)[0], f'open_seed{seed}', tilt_mode='auto_per_sector', seed=seed)
mesh = trimesh.load(os.path.join(SCENE_DIR, 'city.ply'), force='mesh', process=False)
down = mesh.face_normals[:, 2] < -0.5
faces = np.asarray(mesh.faces).copy(); faces[down] = faces[down][:, ::-1]
FLIP_DIR = './scene_open_upfacing'
os.makedirs(FLIP_DIR, exist_ok=True)
trimesh.Trimesh(mesh.vertices, faces, process=False).export(os.path.join(FLIP_DIR, 'city.ply'))
print(f're-wound {down.sum():,} of {len(faces):,} triangles')
cfg = copy.deepcopy(BASE); cfg['MESH_PATH'] = FLIP_DIR; cfg['OUT_DIR'] = os.path.join(OUT_DIR, 'upfacing')
save(T.run(cfg)[0], 'open_terrain_upfacing', tilt_mode='auto_per_sector', terrain='re-wound to face up')

In [ ]:
# 11. Summary of every result in ./out
import glob
for fn in sorted(glob.glob(os.path.join(OUT_DIR, 'open_*.json'))):
    d = json.load(open(fn))
    b = np.array([np.nan if v is None else v for v in d['rsrp_dbm']], dtype=float)
    print(f"{os.path.basename(fn):34} {len(b):4} pts | path {np.isfinite(b).mean()*100:5.1f}% | "
          f"median {np.nanmedian(b):6.1f} dBm | cov>=-90 {(b >= -90).mean()*100:5.1f}%")

In [ ]:
# 12. Map of the full-tile run in ./out (re-run this cell after any run above)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def show_map(fname):
    d = json.load(open(os.path.join(OUT_DIR, fname)))
    n = d['grid']['n']; step = d['grid']['step_m']
    b = np.array([np.nan if v is None else v for v in d['rsrp_dbm']], dtype=float)
    img = np.flipud(b.reshape(n, n))          # rows arrive north->south
    ext = (n - 1) * step / 2
    plt.figure(figsize=(7.5, 6.5))
    plt.imshow(img, origin='lower', extent=[-ext, ext, -ext, ext],
               vmin=-160, vmax=-60, cmap='turbo')
    plt.colorbar(label='Best-server RSRP (dBm per resource element)')
    for name, t in d['towers'].items():
        plt.plot(t['x'], t['y'], marker='*', ms=16, color='white',
                 markeredgecolor='black')
        plt.annotate(name, (t['x'], t['y']), color='white', fontsize=11,
                     weight='bold', ha='center', va='center')
    tilts = ', '.join(f"{s['name']} {s['tilt']:.0f}" for s in d['sectors'])
    plt.title(f"{fname} | tilt mode: {d['run'].get('tilt_mode', '?')}\n"
              f"sector tilts (deg): {tilts}")
    plt.xlabel('East (m)'); plt.ylabel('North (m)')
    plt.tight_layout(); plt.show()
    fin = np.isfinite(b)
    print(f"{fname}: path {fin.mean()*100:.1f}% | median "
          f"{np.nanmedian(b):.1f} dBm | coverage >= -90 dBm "
          f"{(b >= -90).mean()*100:.1f}%")

show_map('open_full_tile.json')


In [ ]:
# 13. Results in the paper's format: Tables II, III and IV and the remaining
# maps (Fig. 2(b) SINR, Fig. 3 tower-B window), computed from the files in
# ./out. Run after the summary cell; runs not yet produced are noted.
import warnings
import matplotlib.pyplot as plt

def _load(fname):
    p = os.path.join(OUT_DIR, fname)
    return json.load(open(p)) if os.path.exists(p) else None

def _b(d):
    return np.array([np.nan if v is None else v for v in d['rsrp_dbm']], dtype=float)

def _row(d):
    b = _b(d)
    return np.isfinite(b).mean() * 100, np.nanmedian(b), (b >= -90).mean() * 100

def _s(x):
    return f"{x:.1f}".replace('-', '\u2212')

def _table(caption, header, rows):
    print(caption)
    print(' | '.join(header))
    for r in rows:
        print(' | '.join(str(x) for x in r))
    print()

# ---- TABLE II ----
d = _load('open_full_tile.json')
w = _load('open_towerB_window_10m.json')
if d is not None:
    rows = []
    p, m, c = _row(d)
    rows.append(['Open city scene, full tile', f'{p:.1f}%', f'{_s(m)} dBm', f'{c:.1f}%'])
    xy = np.array([[cc['x'], cc['y']] for cc in d['cells']], dtype=float)
    bb = _b(d)
    for name, tw in d['towers'].items():
        sel = np.hypot(xy[:, 0] - tw['x'], xy[:, 1] - tw['y']) <= 150.0
        bs = bb[sel]
        rows.append([f'within 150 m of tower {name} ({sel.sum()} points)',
                     f'{np.isfinite(bs).mean() * 100:.1f}%', f'{_s(np.nanmedian(bs))} dBm',
                     f'{(bs >= -90).mean() * 100:.1f}%'])
    sel = (xy[:, 0] >= 120) & (xy[:, 0] <= 380) & (xy[:, 1] >= 120) & (xy[:, 1] <= 380)
    bs = bb[sel]
    rows.append([f'inside the tower-B window ({sel.sum()} points)',
                 f'{np.isfinite(bs).mean() * 100:.1f}%', f'{_s(np.nanmedian(bs))} dBm',
                 f'{(bs >= -90).mean() * 100:.1f}%'])
    if w is not None:
        p, m, c = _row(w)
        seen = {}
        for cc, v in zip(w['cells'], w['rsrp_dbm']):
            seen.setdefault((cc['x'], cc['y']), v)
        u = np.array([np.nan if v is None else v for v in seen.values()], dtype=float)
        rows.append([f'Open city scene, tower-B window, 10 m grid ({len(_b(w))} points; '
                     f'{len(u)} distinct: {np.isfinite(u).mean() * 100:.1f}%, '
                     f'{(u >= -90).mean() * 100:.1f}%)',
                     f'{p:.1f}%', f'{_s(m)} dBm', f'{c:.1f}%'])
    _table('TABLE II. Headline results (street grid points; 3.6 GHz, 47 dBm EIRP; '
           'coverage = best-server RSRP \u2265 \u221290 dBm)',
           ['Scene / run', 'Cells with a path', 'Median best RSRP',
            'Coverage \u2265 \u221290 dBm'], rows)

# ---- TABLE III ----
HEIGHT_BY_SUFFIX = {'E': 20, 'W': 30, 'N': 35, 'S': 50}
rows = []
for case in ['A-E', 'A-W', 'A-N', 'A-S', 'B-E', 'B-W', 'B-N', 'B-S',
             'C-E', 'C-W', 'C-N', 'C-S']:
    dd = _load(f'open_shift_{case}.json')
    if dd is None:
        continue
    tower, suf = case.split('-')
    hgt = dd.get('run', {}).get('heights_m_AGL', {}).get(tower, HEIGHT_BY_SUFFIX[suf])
    p, m, c = _row(dd)
    rows.append([case, f'{float(hgt):g}', f'{p:.1f}%', _s(m), f'{c:.1f}%'])
if rows:
    _table('TABLE III. Tower-shift sweep, open scene (625 cells per case; baseline 24.3%)',
           ['Case', 'Height (m)', 'Cells with path', 'Median RSRP (dBm)',
            'Coverage \u2265 \u221290 dBm'], rows)

# ---- TABLE IV ----
rows = []
for lab, f in [('Baseline (auto tilt, ground-referenced)', 'open_full_tile.json'),
               ('Fixed tilt (4\u00b0, ground-referenced)', 'open_tilt4.json'),
               ('Roof-referenced (roof + 10 m, 4\u00b0 fixed; raises A, C, lowers B)',
                'open_tilt4_roofref.json'),
               ('Roof-referenced, solver depth 5', 'open_tilt4_roofref_depth5.json')]:
    dd = _load(f)
    if dd is None:
        continue
    p, m, c = _row(dd)
    rows.append([lab, f'{p:.1f}%', f'{_s(m)} dBm', f'{c:.1f}%'])
if rows:
    _table('TABLE IV. Sensitivity of the headline result to configuration '
           '(625 street cells, 40 m grid; all runs 3.6 GHz, EIRP 47 dBm)',
           ['Configuration', 'Cells with a path', 'Median best RSRP',
            'Coverage \u2265 \u221290 dBm'], rows)

# ---- Fig. 2(b) SINR map and Fig. 3 tower-B window map ----
def _grid(d):
    g = d['grid']
    if 'n_x' in g:
        return g['n_x'], g['n_y'], g['x_m'], g['y_m']
    n = g['n']
    half = (n - 1) * g['step_m'] / 2
    return n, n, [-half, half], [-half, half]

def _show(mat, d, vmin, vmax, cmap, label, title, towers):
    nx, ny, xm, ym = _grid(d)
    img = np.flipud(np.array(mat, dtype=float).reshape(ny, nx))
    plt.figure(figsize=(6, 5))
    plt.imshow(img, origin='lower', extent=[xm[0], xm[1], ym[0], ym[1]],
               vmin=vmin, vmax=vmax, cmap=cmap)
    plt.colorbar(label=label)
    for t, lab in towers:
        tw = d['towers'][t]
        plt.plot(tw['x'], tw['y'], marker='*', color='red', markersize=14,
                 markeredgecolor='black', linestyle='')
        plt.text(tw['x'] + 18, tw['y'] + 18, lab, color='red', fontsize=11,
                 fontweight='bold')
    plt.xlabel('East (m)')
    plt.ylabel('North (m)')
    if title:
        plt.title(title)
    plt.show()

if d is not None:
    P = np.array([[np.nan if v is None else 10 ** (v / 10) for v in row]
                  for row in d['rsrp']], dtype=float)  # mW per sector
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sig = np.nanmax(P, axis=1)
        inter = np.nansum(P, axis=1) - sig
    sinr = 10 * np.log10(sig / (inter + 10 ** (-122.2 / 10)))  # noise per RE, NF 7 dB
    sinr[~np.isfinite(_b(d))] = np.nan
    print(f'Fig. 2(b): median best-server SINR over points with a path = '
          f'{np.nanmedian(sinr):.1f} dB')
    _show(sinr, d, -20, 30, 'RdYlBu', 'Best-server SINR (dB)',
          '(b) SINR, full load, NF 7 dB',
          [('A', 'A'), ('B', 'B'), ('C', 'C')])
if w is not None:
    print('Fig. 3. Urban coverage around tower B: best-server RSRP '
          '(dBm per resource element) on a 10 m grid over a 260 x 260 m '
          'window centred on the site (star; white: no path).')
    _show(w['rsrp_dbm'], w, -160, -60, 'turbo',
          'Best-server RSRP (dBm per resource element)', None,
          [('B', 'Tower B')])
